# DSA508 Problem 1: Exploring sample_mflix with MongoDB
`sample_mflix` is MongoDB's demo database for a movie website. The movies come from IMDb and Rotten Tomatoes, while the users, comments, theaters and sessions were made up to support a demo app. In this notebook I treat it the way an analyst would treat a new client database. First I find out what is in it and how trustworthy each part is (part a). Then I look for patterns over time and across genres, and in the language of the plots (part b). Finally I use topic modeling to recover the story themes that sit underneath the genre labels (part c).

All aggregation happens in MongoDB through MQL pipelines (`$match`, `$group`, `$unwind`, `$bucket`, `$lookup`, `$sortByCount`, `$median`). Python only handles the charts and the NLP models. Set the `MONGO_URI` environment variable to your Atlas connection string before running. The figures quoted in the text come from my run. The Atlas copy of the dataset is a slightly newer revision, so exact counts may differ a little, but the patterns hold.

In [ ]:
import os, re, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from matplotlib.colors import LinearSegmentedColormap, ListedColormap
from matplotlib.gridspec import GridSpec
from scipy.stats import gaussian_kde
import plotly.graph_objects as go, plotly.io as pio
from pymongo import MongoClient
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.decomposition import NMF, LatentDirichletAllocation
from sklearn.manifold import TSNE

# connection string comes from env var so no password lives in the notebook
URI = os.getenv("MONGO_URI", "mongodb://127.0.0.1:27017")
db = MongoClient(URI)["sample_mflix"]
pd.set_option("display.width", 140)
pio.renderers.default = "plotly_mimetype+notebook_connected"

# one visual system for every chart: ivory surface, a single blue ramp, blue-gray-red for diverging values
IVORY, INK, INK2, MUTED, GRID, TRACK, RED = "#f7f4ec", "#1f2a37", "#52514e", "#898781", "#e6e3da", "#e3e6ea", "#e34948"
B = {100: "#cde2fb", 150: "#b7d3f6", 200: "#9ec5f4", 250: "#86b6ef", 300: "#6da7ec", 350: "#5598e7", 400: "#3987e5",
     450: "#2a78d6", 500: "#256abf", 550: "#1c5cab", 600: "#184f95", 650: "#104281", 700: "#0d366b"}
CAT = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]  # validated categorical order
BLUES = LinearSegmentedColormap.from_list("blues", [B[100], B[300], B[500], B[700]])
DIV = LinearSegmentedColormap.from_list("div", [RED, "#f0efec", B[550]])
plt.rcParams.update({"figure.facecolor": IVORY, "axes.facecolor": IVORY, "savefig.facecolor": IVORY, "figure.dpi": 110,
                     "axes.edgecolor": GRID, "axes.grid": True, "grid.color": GRID, "grid.linewidth": .8, "axes.axisbelow": True,
                     "axes.spines.top": False, "axes.spines.right": False, "axes.spines.left": False,
                     "xtick.color": MUTED, "ytick.color": MUTED, "xtick.labelsize": 9, "ytick.labelsize": 9,
                     "axes.labelcolor": INK2, "axes.labelsize": 9.5, "text.color": INK, "font.size": 10, "legend.frameon": False})

# left-aligned title with a muted subtitle, used on every chart
def head(ax, title, sub=""):
    ax.text(0, 1.11, title, transform=ax.transAxes, fontsize=13, fontweight="bold", color=INK, va="bottom")
    ax.text(0, 1.03, sub, transform=ax.transAxes, fontsize=9.5, color=MUTED, va="bottom")

# run an aggregation pipeline and return a DataFrame
def agg(coll, pipeline):
    return pd.DataFrame(list(db[coll].aggregate(pipeline)))

## Part (a): Exploratory data analysis
### 1. What is in the database?

In [ ]:
# document count and field count of a sample doc, per collection
rows = []
for c in sorted(db.list_collection_names()):
    doc = db[c].find_one() or {}
    rows.append({"collection": c, "documents": db[c].estimated_document_count(), "top_level_fields": len(doc)})
pd.DataFrame(rows)

**Reading the structure.** `movies` is the core collection, with about 23.5k documents. `comments` is the largest, at about 50k documents, and links to movies through `movie_id`, which works like a foreign key. `users` is small (185 accounts), `theaters` holds about 1.5k US cinema locations with GeoJSON points, and `sessions` holds a single login token. So this is really a movie catalogue with a thin, simulated user layer on top. That framing matters for everything that follows.

### 2. Schema and field coverage in `movies`

In [ ]:
# presence flags computed in MongoDB for a random sample, then drawn as a barcode (one column per movie)
fields = ["title","year","genres","countries","awards.wins","imdb.rating","imdb.votes","languages","directors","cast","runtime",
          "plot","fullplot","tomatoes.viewer.rating","num_mflix_comments","rated","tomatoes.critic.rating","metacritic"]
has = lambda f: {"$cond": [{"$and": [{"$ne": [{"$ifNull": [f"${f}", None]}, None]}, {"$ne": [f"${f}", ""]}]}, 1, 0]}
samp = agg("movies", [{"$sample": {"size": 700}}, {"$project": {"_id": 0, "yr": "$year", **{f.replace(".", "_"): has(f) for f in fields}}}])
samp = samp.assign(y=pd.to_numeric(samp.yr, errors="coerce")).sort_values("y")
M = samp[[f.replace(".", "_") for f in fields]].T.values
total = db.movies.estimated_document_count()
cov = {f: db.movies.count_documents({f: {"$exists": True, "$nin": [None, "", []]}}) / total for f in fields}

fig, ax = plt.subplots(figsize=(13, 5.6))
ax.imshow(M, aspect="auto", cmap=ListedColormap([TRACK, B[450]]), interpolation="nearest")
ax.set_yticks(range(len(fields)), fields); ax.set_xticks([]); ax.grid(False)
for i, f in enumerate(fields):
    ax.text(M.shape[1] + 8, i, f"{cov[f]:.0%}", va="center", fontsize=9, color=B[650] if cov[f] > .9 else RED if cov[f] < .6 else INK2)
yrs = samp.y.values
for q in [0, .25, .5, .75, .999]:
    j = int(q * (len(yrs) - 1)); ax.text(j, len(fields) + .3, f"{int(yrs[j]) if yrs[j] == yrs[j] else ''}", ha="center", va="top", fontsize=8.5, color=MUTED)
head(ax, "Field coverage barcode", "700 random movies sorted oldest to newest, blue = field present, right = coverage across all movies")
plt.tight_layout(); plt.show()

In [ ]:
# mixed BSON types are a classic document-database issue, so check the key numeric fields
for f in ["year", "imdb.rating", "runtime"]:
    print(f, agg("movies", [{"$group": {"_id": {"$type": f"${f}"}, "n": {"$sum": 1}}}]).set_index("_id").n.to_dict())
print(db.movies.find({"year": {"$type": "string"}}, {"_id": 0, "title": 1, "year": 1}).limit(3).to_list())
# duplicate documents: same title and year stored more than once
dup = agg("movies", [{"$group": {"_id": {"t": "$title", "y": "$year"}, "n": {"$sum": 1}}}, {"$match": {"n": {"$gt": 1}}}])
print(f"title/year pairs stored more than once: {len(dup)} ({dup.n.sum()} documents)")

**Data quality.** Movie documents are rich (18 top-level fields, with nested `imdb`, `tomatoes` and `awards` sub-documents and arrays for cast, genres and countries), but coverage is uneven. IMDb data is close to complete, Rotten Tomatoes audience scores cover about 87% of movies, critic scores only about half, and Metacritic a third. The MPAA `rated` field is missing for 42% of movies, mostly older and foreign films that were never submitted for a US rating.

The schema-less design also shows up in the types. 37 movies store `year` as a string with a stray character (`"1981è"`), 61 have an empty string as their IMDb rating, and about 750 title/year pairs appear more than once. None of this would get past a relational schema. It's a good reminder that in a document store, validation is the analyst's job. Every pipeline below filters with `$type: "number"` / `"int"` so that bad values don't silently skew the averages, and the text analysis removes the duplicates. In the barcode, the core fields are solid bands, while the critic score, Metacritic and MPAA rating rows are speckled with gaps across every era.

### 3. Descriptive statistics for movies

In [ ]:
# central tendency and spread for IMDb ratings, votes and runtime
stats = agg("movies", [
    {"$match": {"imdb.rating": {"$type": "number"}}},
    {"$group": {"_id": None, "movies": {"$sum": 1},
                "rating_mean": {"$avg": "$imdb.rating"}, "rating_sd": {"$stdDevPop": "$imdb.rating"},
                "rating_median": {"$median": {"input": "$imdb.rating", "method": "approximate"}},
                "votes_mean": {"$avg": "$imdb.votes"}, "votes_median": {"$median": {"input": "$imdb.votes", "method": "approximate"}},
                "runtime_mean": {"$avg": "$runtime"}, "runtime_median": {"$median": {"input": "$runtime", "method": "approximate"}},
                "first_year": {"$min": "$year"}}}])
stats.drop(columns="_id").T.round(2)

In [ ]:
# reach vs quality as a hexbin with marginal distributions
d = agg("movies", [{"$match": {"imdb.rating": {"$type": "number"}, "imdb.votes": {"$type": "number", "$gt": 0}}},
                   {"$project": {"_id": 0, "rating": "$imdb.rating", "votes": "$imdb.votes"}}])
d["lv"] = np.log10(d.votes)
fig = plt.figure(figsize=(11, 7.5)); gs = GridSpec(4, 4, figure=fig, hspace=.06, wspace=.06)
ax = fig.add_subplot(gs[1:, :3]); top = fig.add_subplot(gs[0, :3], sharex=ax); side = fig.add_subplot(gs[1:, 3], sharey=ax)
hb = ax.hexbin(d.lv, d.rating, gridsize=48, cmap=BLUES, mincnt=1, bins="log", linewidths=.2, edgecolors=IVORY)
ax.set_xticks([1, 2, 3, 4, 5, 6], ["10", "100", "1k", "10k", "100k", "1M"]); ax.set_xlabel("IMDb votes (log scale)"); ax.set_ylabel("IMDb rating")
top.hist(d.lv, bins=60, color=B[400], edgecolor=IVORY, linewidth=.4); side.hist(d.rating, bins=60, color=B[400], orientation="horizontal", edgecolor=IVORY, linewidth=.4)
for a in (top, side): a.axis("off")
ax.axhline(d.rating.mean(), color=RED, lw=1); ax.text(.75, d.rating.mean() + .12, f"mean {d.rating.mean():.2f}", color=RED, fontsize=9)
ax.annotate("blockbusters: many votes,\nratings pulled to 6 to 8", xy=(5.6, 7.2), xytext=(4.7, 2.6), fontsize=9, color=INK2,
            arrowprops=dict(arrowstyle="-", color=MUTED, lw=.8))
head(top, "Reach vs quality", f"{len(d):,} movies; darker hexes hold more films. Ratings fan out when few people vote and converge as votes grow")
plt.show()

**The typical movie.** Ratings are tightly clustered: the mean IMDb score is 6.7 with a standard deviation of about 1.0, and the distribution is skewed left (median 6.8), with a long tail of poorly rated films and very few above 8.5. Votes are the opposite. The median movie has about 2,000 votes but the mean is about 20,000, because a few blockbusters gather hundreds of thousands of votes. That's why votes are plotted on a log scale and why medians are more honest than means here. The hexbin shows a clear fan shape: films with few votes have ratings spread across the whole scale, while films with hundreds of thousands of votes converge on 6 to 8, because a big audience averages out extreme opinions. The typical runtime is 100 minutes, with outliers from one-minute 1890s shorts to multi-hour miniseries.

### 4. Catalogue composition: genres, countries, languages, ratings

In [ ]:
# UpSet plot: the most common exact genre combinations, computed with $sortArray + $group
combos = agg("movies", [{"$match": {"genres.0": {"$exists": True}}},
                        {"$group": {"_id": {"$sortArray": {"input": "$genres", "sortBy": 1}}, "n": {"$sum": 1}}},
                        {"$sort": {"n": -1}}, {"$limit": 16}])
gset = pd.Series([g for c in combos._id for g in c]).value_counts().index.tolist()
fig = plt.figure(figsize=(13, 6.8)); gs = GridSpec(2, 1, height_ratios=[1.5, 1], hspace=.04, figure=fig)
ax, mx = fig.add_subplot(gs[0]), fig.add_subplot(gs[1])
ax.bar(range(len(combos)), combos.n, width=.45, color=[B[650] if len(c) == 1 else B[400] for c in combos._id])
for i, v in enumerate(combos.n): ax.text(i, v + 40, f"{v:,}", ha="center", fontsize=8.5, color=INK2)
ax.set_xticks([]); ax.set_xlim(-.6, len(combos) - .4); ax.set_ylabel("movies")
for i, c in enumerate(combos._id):
    ys = [gset.index(g) for g in c]
    mx.scatter([i] * len(gset), range(len(gset)), s=70, color=TRACK, zorder=1)
    mx.scatter([i] * len(ys), ys, s=70, color=B[650], zorder=3)
    if len(ys) > 1: mx.plot([i, i], [min(ys), max(ys)], color=B[650], lw=2, zorder=2)
mx.set_yticks(range(len(gset)), gset); mx.invert_yaxis(); mx.set_xticks([]); mx.set_xlim(-.6, len(combos) - .4); mx.grid(False)
head(ax, "Genre combinations (UpSet plot)", "each column is an exact genre set; dark bars are single-genre films. Drama is the hub almost every combination passes through")
plt.show()

In [ ]:
# where films come from and how they are rated, as lollipops
countries = agg("movies", [{"$unwind": "$countries"}, {"$sortByCount": "$countries"}, {"$limit": 10}])
rated = agg("movies", [{"$sortByCount": {"$ifNull": ["$rated", "missing"]}}, {"$limit": 8}])
n_genres = agg("movies", [{"$group": {"_id": {"$size": {"$ifNull": ["$genres", []]}}, "movies": {"$sum": 1}}}, {"$sort": {"_id": 1}}])
fig, axs = plt.subplots(1, 2, figsize=(13, 4.6))
for ax, t, sub, dd in [(axs[0], "Top production countries", "movies listing each country (a film can list several)", countries),
                       (axs[1], "MPAA rating", "missing is the largest group: older and foreign films were never rated", rated)]:
    dd = dd.iloc[::-1]
    ax.hlines(dd._id, 0, dd["count"], color=B[200], lw=2.5); ax.scatter(dd["count"], dd._id, s=60, color=[RED if x == "missing" else B[550] for x in dd._id], zorder=3)
    for y, v in zip(dd._id, dd["count"]): ax.text(v + dd["count"].max() * .02, y, f"{v:,}", va="center", fontsize=8.5, color=INK2)
    ax.grid(axis="y", visible=False); head(ax, t, sub)
plt.tight_layout(); plt.show()
print(n_genres.rename(columns={"_id": "genres_per_movie"}).to_string(index=False))

**What the catalogue is made of.** Drama dominates and appears on more than half of all movies, followed by Comedy and Romance. Most movies carry two or three genres (IMDb caps this at three), so genre is a set of labels rather than one category, which is why every genre pipeline starts with `$unwind`. The UpSet plot shows the exact combinations: Drama on its own is by far the most common set, and Drama appears in most of the top combinations, so it works more like a default label than a distinct genre. The catalogue is heavily American (about half of all movies list the USA), with France, the UK, Germany and Italy forming a strong European second tier, and English plus four European languages account for most of the spoken languages. Any conclusion here describes Western, mostly Hollywood, film culture rather than world cinema.

### 5. Do critics and audiences agree?

In [ ]:
# compare four rating sources on the movies that have all of them
r = agg("movies", [
    {"$match": {"imdb.rating": {"$type": "number"}, "tomatoes.critic.rating": {"$type": "number"},
                "tomatoes.viewer.rating": {"$type": "number"}, "metacritic": {"$type": "number"}}},
    {"$project": {"_id": 0, "IMDb": "$imdb.rating", "RT critics": "$tomatoes.critic.rating",
                  "RT audience": "$tomatoes.viewer.rating", "Metacritic": "$metacritic"}}])
c = r.corr().iloc[1:, :-1]; mask = np.triu(np.ones_like(c, dtype=bool))
fig, ax = plt.subplots(figsize=(6.6, 5.2))
sns.heatmap(c, mask=mask, annot=True, fmt=".2f", cmap=BLUES, vmin=.4, vmax=1, square=True, linewidths=3, linecolor=IVORY,
            cbar=False, annot_kws={"fontsize": 12, "fontweight": "bold"}, ax=ax)
ax.grid(False); ax.tick_params(length=0)
head(ax, "Two rating camps", f"Pearson r on {len(r):,} movies with all four scores")
plt.show()

**Critics vs audiences.** The four rating systems split into two camps. Rotten Tomatoes critics and Metacritic agree almost perfectly (r = 0.92) because both summarise professional reviews. IMDb lines up best with the Rotten Tomatoes audience score (0.80) because both are crowd ratings. Across the two camps, critics and audiences only correlate at 0.65. In other words, "is this a good movie?" has two fairly different answers depending on who you ask. Part (b) looks at which genres drive that disagreement.

### 6. Awards and the most prolific directors

In [ ]:
# awards coverage and the directors with the most films in the catalogue
aw = agg("movies", [{"$group": {"_id": None, "with_win": {"$sum": {"$cond": [{"$gt": ["$awards.wins", 0]}, 1, 0]}},
                                 "avg_wins": {"$avg": "$awards.wins"}, "max_wins": {"$max": "$awards.wins"}}}])
print(aw.drop(columns="_id").round(2).to_string(index=False))
agg("movies", [{"$unwind": "$directors"}, {"$match": {"imdb.rating": {"$type": "number"}}},
               {"$group": {"_id": "$directors", "films": {"$sum": 1}, "avg_rating": {"$avg": "$imdb.rating"}}},
               {"$sort": {"films": -1}}, {"$limit": 10}]).round(2)

### 7. Users, comments and engagement

In [ ]:
# basic engagement numbers and how comments link users to movies
per_user = agg("comments", [{"$group": {"_id": "$email", "n": {"$sum": 1}}}])
per_movie = agg("comments", [{"$group": {"_id": "$movie_id", "n": {"$sum": 1}}}])
print(f"comments: {db.comments.estimated_document_count()}, distinct commenters: {len(per_user)}, registered users: {db.users.estimated_document_count()}")
print(f"movies with at least one comment: {len(per_movie)}")
print("comments per user:", per_user.n.describe().round(1).to_dict())
print("comments per movie:", per_movie.n.describe().round(1).to_dict())
# top commented movies, joined back to movies with $lookup
agg("comments", [{"$sortByCount": "$movie_id"}, {"$limit": 5},
                 {"$lookup": {"from": "movies", "localField": "_id", "foreignField": "_id", "as": "m"}},
                 {"$project": {"_id": 0, "comments": "$count", "title": {"$first": "$m.title"},
                               "year": {"$first": "$m.year"}, "imdb": {"$first": "$m.imdb.rating"}}}])

In [ ]:
# a 24-hour clock: rings are weekdays, angle is the hour. Real platforms show a night-time dip; this one is flat
wh = agg("comments", [{"$group": {"_id": {"d": {"$dayOfWeek": "$date"}, "h": {"$hour": "$date"}}, "n": {"$sum": 1}}}])
grid = np.zeros((7, 24))
for _, r_ in wh.iterrows(): grid[r_._id["d"] - 1, r_._id["h"]] = r_.n
by_year = agg("comments", [{"$group": {"_id": {"$year": "$date"}, "n": {"$sum": 1}}}, {"$sort": {"_id": 1}}])
fig = plt.figure(figsize=(14, 5.6))
ax = fig.add_subplot(1, 2, 1, projection="polar")
theta = np.linspace(0, 2 * np.pi, 25); rad = np.arange(2, 10)
pc = ax.pcolormesh(theta, rad, grid, cmap=BLUES, vmin=0, vmax=grid.max(), edgecolors=IVORY, linewidth=1.2)
ax.set_theta_zero_location("N"); ax.set_theta_direction(-1); ax.set_yticks([]); ax.grid(False); ax.spines["polar"].set_visible(False)
ax.set_xticks(np.linspace(0, 2 * np.pi, 24, endpoint=False) + np.pi / 24, [f"{h:02d}" for h in range(24)], fontsize=8, color=MUTED)
for i, dn in enumerate(["Sun", "Mon", "Tue", "Wed", "Thu", "Fri", "Sat"]): ax.text(np.pi / 2 * 0, 2.5 + i, dn, fontsize=7, color=INK2, ha="center", va="center")
ax.set_title(f"Comments by weekday (rings, Sun inside) x hour\nmin {grid.min():.0f} / max {grid.max():.0f} per cell, almost no rhythm", fontsize=11, color=INK, pad=18)
ax2 = fig.add_subplot(1, 2, 2)
ax2.bar(by_year._id, by_year.n, width=.5, color=B[400]); ax2.set_ylabel("comments")
head(ax2, "Comments per year", "flat at ~1,050 a year since 1970, decades before online reviews existed")
plt.tight_layout(); plt.show()

In [ ]:
# sanity check: can a comment be written before the movie came out?
chk = agg("comments", [
    {"$lookup": {"from": "movies", "localField": "movie_id", "foreignField": "_id", "as": "m"}}, {"$unwind": "$m"},
    {"$match": {"m.year": {"$type": "int"}}},
    {"$group": {"_id": None, "n": {"$sum": 1}, "before_release": {"$sum": {"$cond": [{"$lt": [{"$year": "$date"}, "$m.year"]}, 1, 0]}}}}])
pct = chk.before_release[0] / chk.n[0]
# waffle chart: 100 squares, each 1% of comments
fig, ax = plt.subplots(figsize=(8.5, 3))
k = int(round(pct * 100))
for i in range(100):
    ax.add_patch(plt.Rectangle((i // 10 * 1.1, 9 - i % 10 * 1.0 * 1.0), .9, .82, color=B[650] if i < k else TRACK))
ax.set_xlim(-.2, 11 + 7); ax.set_ylim(-.3, 10.2); ax.axis("off"); ax.set_aspect("equal")
ax.text(12, 6.2, f"{pct:.0%}", fontsize=44, fontweight="bold", color=B[650])
ax.text(12, 3.6, f"of {chk.n[0]:,} comments are dated\nbefore the movie's release year", fontsize=11, color=INK2)
plt.show()
# does comment volume track popularity?
eng = agg("movies", [{"$match": {"num_mflix_comments": {"$gt": 0}, "imdb.votes": {"$type": "number"}, "imdb.rating": {"$type": "number"}}},
                     {"$project": {"_id": 0, "comments": "$num_mflix_comments", "imdb_votes": "$imdb.votes", "imdb_rating": "$imdb.rating"}}])
eng.corr(method="spearman").round(3)

In [ ]:
# a look at the comment text itself
for t in db.comments.aggregate([{"$sample": {"size": 3}}, {"$project": {"_id": 0, "text": 1}}]):
    print("-", t["text"][:110])

**The engagement data is synthetic, and that is itself a finding.** On the surface the numbers look reasonable: about 50k comments from 315 commenters on about 14k movies. Looking closer, nothing behaves like real users:

- Comment volume is flat at roughly 1,050 a year from 1970 to 2016, decades before online movie reviews existed. The clock chart shows it is also flat across every weekday and hour, with no night-time dip. Real activity has weekly and daily rhythms.
- **63% of comments are dated before the movie's release year** (the waffle chart), which is impossible for real reviews.
- Comment count has essentially zero correlation with IMDb votes or rating (Spearman ρ ≈ 0.01 to 0.03). Real attention follows popularity.
- The users are *Game of Thrones* characters and actors with fake email domains, and the comment text is *lorem ipsum* placeholder Latin.

So the comments are generated filler meant to test an app, not evidence of audience behaviour. I still use them to demonstrate joins (`$lookup` from comments to movies) but do **not** draw conclusions about what audiences like from them. For audience signal I rely on IMDb votes and the Rotten Tomatoes audience score, which come from real people. The same reasoning applies to the topic modeling in part (c), which runs on movie plots rather than comment text.

### 8. Theaters and sessions

In [ ]:
# theater footprint by state, plus a scatter of the GeoJSON points
th = agg("theaters", [{"$project": {"_id": 0, "state": "$location.address.state",
                                    "lon": {"$arrayElemAt": ["$location.geo.coordinates", 0]},
                                    "lat": {"$arrayElemAt": ["$location.geo.coordinates", 1]}}}])
fig, axs = plt.subplots(1, 2, figsize=(14, 5), gridspec_kw={"width_ratios": [2.2, 1]})
us = th[(th.lon > -128) & (th.lon < -65) & (th.lat > 24) & (th.lat < 50)]
axs[0].hexbin(us.lon, us.lat, gridsize=42, cmap=BLUES, mincnt=1, linewidths=.3, edgecolors=IVORY)
axs[0].set_aspect(1.25); axs[0].set_xticks([]); axs[0].set_yticks([]); axs[0].grid(False)
for s_ in axs[0].spines.values(): s_.set_visible(False)
head(axs[0], "A map drawn only from GeoJSON points", "theater density, contiguous US; the country's outline emerges from the data")
st_ = th.state.value_counts().head(10).iloc[::-1]
axs[1].hlines(st_.index, 0, st_.values, color=B[200], lw=2.5); axs[1].scatter(st_.values, st_.index, s=55, color=B[550], zorder=3)
axs[1].grid(axis="y", visible=False); head(axs[1], "Top states", "theaters per state")
plt.tight_layout(); plt.show()
print(f"theaters: {len(th)}, states: {th.state.nunique()}, sessions: {db.sessions.estimated_document_count()}")

**Theaters.** The 1,564 theaters span all states, and their counts roughly follow population (California, Texas, Florida, New York), and the hexbin density map traces the outline of the continental US from the coordinates alone, with no basemap. The locations are stored as GeoJSON points, so the collection is set up for `$geoNear` queries such as "theaters within 10 miles of me" in the demo app. It holds no showtimes or sales, though, so it can't be linked to movie performance.

## Part (b): Temporal and categorical patterns
### 1. Ratings across the decades

In [ ]:
# volume and mean rating per decade with $bucket
dec = agg("movies", [{"$match": {"year": {"$type": "int"}, "imdb.rating": {"$type": "number"}}},
                     {"$bucket": {"groupBy": "$year", "boundaries": list(range(1890, 2031, 10)),
                                  "output": {"movies": {"$sum": 1}, "avg_rating": {"$avg": "$imdb.rating"}, "avg_votes": {"$avg": "$imdb.votes"}}}}])
dec = dec.rename(columns={"_id": "decade"})
# ridgeline: the full rating distribution per decade, ridge color = decade median
rd = agg("movies", [{"$match": {"year": {"$type": "int", "$gte": 1920}, "imdb.rating": {"$type": "number"}}},
                    {"$project": {"_id": 0, "dec": {"$subtract": ["$year", {"$mod": ["$year", 10]}]}, "r": "$imdb.rating"}}])
decs = sorted(rd.dec.unique()); xs = np.linspace(1, 10, 300)
meds = rd.groupby("dec").r.median(); norm = plt.Normalize(meds.min(), meds.max())
fig, ax = plt.subplots(figsize=(12, 7.5))
for i, dd in enumerate(decs[::-1]):
    v = rd[rd.dec == dd].r; y = gaussian_kde(v)(xs); y = y / y.max() * 1.6; base_ = i * 1.0
    ax.fill_between(xs, base_, base_ + y, color=BLUES(norm(meds[dd])), alpha=.95, zorder=len(decs) - i, lw=0)
    ax.plot(xs, base_ + y, color=IVORY, lw=1.6, zorder=len(decs) - i)
    ax.text(.85, base_ + .18, f"{dd}s", ha="right", fontsize=10, color=INK, fontweight="bold")
    ax.text(10.15, base_ + .18, f"n={len(v):,}  median {v.median():.1f}", fontsize=8.5, color=MUTED)
ax.axvline(rd.r.mean(), color=RED, lw=1, zorder=50); ax.set_yticks([]); ax.grid(axis="y", visible=False); ax.set_xlim(0, 12.6); ax.set_xticks(range(1, 11))
ax.set_xlabel("IMDb rating")
head(ax, "Ratings by decade, as a ridgeline", "each ridge is one decade's distribution; darker = higher median; red line = all-time mean. Ridges slide left as catalogues fill up")
plt.show()
dec.round(2)

**Older films score higher, and the reason is survivorship.** Average ratings fall steadily from about 7.3 in the 1930s to 1950s to about 6.5 in the 2010s, while the number of films per decade grows sharply until the 2000s. The ridgeline shows that the whole distribution moves, not just the average: before 1970 the ridges are narrow and centred around 7.3, while from the 1980s they widen and grow a long tail of low-rated films. It's tempting to read this as "movies got worse", but the more likely explanation is selection. A 1940s film is in this database because it was remembered, restored and catalogued, so the forgettable films of that era have mostly dropped out. Recent decades include everything, including the weak releases. On top of that, people who search out and rate an old film are usually fans. Comparisons across decades in this data say more about which films survive in the catalogue than about the quality of a whole era.

### 2. Rating by genre and decade

In [ ]:
# genre x decade pivot for the main genres, unwinding the genres array first
top = ["Drama", "Comedy", "Romance", "Crime", "Thriller", "Action", "Documentary", "Horror", "Sci-Fi", "War", "Western", "Animation"]
gd = agg("movies", [{"$match": {"year": {"$type": "int", "$gte": 1930}, "imdb.rating": {"$type": "number"}}},
                    {"$unwind": "$genres"}, {"$match": {"genres": {"$in": top}}},
                    {"$group": {"_id": {"g": "$genres", "d": {"$subtract": ["$year", {"$mod": ["$year", 10]}]}},
                                "r": {"$avg": "$imdb.rating"}, "n": {"$sum": 1}}}])
gd["genre"], gd["decade"] = gd._id.str["g"], gd._id.str["d"]
gd = gd[gd.n >= 10]
fig, ax = plt.subplots(figsize=(12, 5.6))
mean_r = gd.r.mean()
sns.heatmap(gd.pivot(index="genre", columns="decade", values="r").loc[top], annot=True, fmt=".1f", cmap=DIV, center=6.7,
            vmin=5.6, vmax=7.8, linewidths=3, linecolor=IVORY, cbar=False, annot_kws={"fontsize": 9}, ax=ax)
ax.grid(False); ax.tick_params(length=0); ax.set_xlabel(""); ax.set_ylabel("")
head(ax, "Rating by genre and decade", "blue = above the catalogue average of 6.7, red = below, gray = average; cells with fewer than 10 films left blank")
plt.show()

In [ ]:
# share of each genre within a decade, to see what the industry made more of
share = agg("movies", [{"$match": {"year": {"$type": "int", "$gte": 1930}}}, {"$unwind": "$genres"},
                       {"$group": {"_id": {"g": "$genres", "d": {"$subtract": ["$year", {"$mod": ["$year", 10]}]}}, "n": {"$sum": 1}}}])
share["genre"], share["decade"] = share._id.str["g"], share._id.str["d"]
p = share.pivot(index="decade", columns="genre", values="n").fillna(0)
keep = ["Drama", "Comedy", "Romance", "Crime", "Thriller", "Action", "Documentary"]
st_ = p[keep]
# streamgraph: a stacked area around a wiggling baseline, so growth and mix read together
fig, ax = plt.subplots(figsize=(13, 6))
xs = st_.index.values; cols = CAT[:7]
polys = ax.stackplot(xs, st_.T.values, baseline="sym", colors=cols, edgecolor=IVORY, linewidth=1.2)
ys = np.cumsum(np.vstack([np.zeros(len(xs)), st_.T.values]), axis=0); off = -ys[-1] / 2
for i, g in enumerate(st_.columns):
    j = len(xs) - 2 if g != "Documentary" else len(xs) - 1; mid = off[j] + (ys[i][j] + ys[i + 1][j]) / 2
    if st_[g].iloc[j] > 200: ax.text(xs[j], mid, g, fontsize=9, color="white" if i in (0, 6) else INK, ha="center", va="center", fontweight="bold")
ax.set_yticks([]); ax.grid(axis="y", visible=False); ax.set_xticks(xs, [f"{x}s" for x in xs])
head(ax, "The catalogue as a streamgraph", "the seven largest genres, film counts per decade (a film counts once per genre); stream thickness = number of films")
plt.show()

# bump chart: genre rank by decade, three stories highlighted
rk = p[[g for g in p.columns if p[g].sum() >= 250]].rank(axis=1, ascending=False, method="first")
hi = {"Documentary": CAT[0], "Horror": CAT[1], "Western": CAT[2], "Musical": CAT[6]}
fig, ax = plt.subplots(figsize=(13, 6.2))
for g in rk.columns:
    c_, lw, z = (hi[g], 3, 3) if g in hi else ("#cfccc3", 1.4, 1)
    ax.plot(rk.index, rk[g], color=c_, lw=lw, zorder=z, marker="o", ms=6 if g in hi else 4, mec=IVORY)
    ax.text(rk.index[-1] + 2, rk[g].iloc[-1], g, va="center", fontsize=9, color=c_ if g in hi else MUTED, fontweight="bold" if g in hi else None)
ax.invert_yaxis(); ax.set_yticks(range(1, int(rk.max().max()) + 1)); ax.set_ylabel("rank by number of films"); ax.grid(axis="x", visible=False)
ax.set_xticks(rk.index, [f"{x}s" for x in rk.index]); ax.set_xlim(rk.index[0] - 2, rk.index[-1] + 14)
head(ax, "Genre ranks over time (bump chart)", "Documentary climbs into the top three; Westerns and Musicals fall away; Horror rises in the 1970s and fades")
plt.show()

**Genre trends.** The heatmap shows that the decline is not the same everywhere. **Horror** falls the furthest, from 7.0 to 7.4 before 1970 to 5.6 in the 2010s, which fits the slasher and low-budget boom from the 1970s on: lots of cheap films aimed at a narrow audience. **Thriller** and **Sci-Fi** drop in a similar way. **Documentary** and **War** stay high in every decade.

The streamgraph shows how sharply the catalogue grows after 1980, and the bump chart shows how the genre rankings reshuffle. Documentary climbs from near the bottom into the top three by the 2010s, Horror rises in the 1970s and then slips back, and Westerns, musicals and film noir effectively disappear after the 1960s, war films peak around WWII and then shrink, and Documentary grows from about 1% of releases to almost 8% in the 2010s, helped by cheaper digital production and streaming. Drama is consistently a quarter to a third of everything.

### 3. Genre profile: rating vs popularity, and the critic/audience gap

In [ ]:
# per-genre mean rating vs mean votes, bubble size = number of films
gp = agg("movies", [{"$match": {"imdb.rating": {"$type": "number"}}}, {"$unwind": "$genres"},
                    {"$group": {"_id": "$genres", "films": {"$sum": 1}, "rating": {"$avg": "$imdb.rating"}, "votes": {"$avg": "$imdb.votes"}}},
                    {"$match": {"films": {"$gte": 100}}}])
fig, ax = plt.subplots(figsize=(12, 6.5))
hl = {"Documentary": B[650], "Horror": RED, "Sci-Fi": B[650], "Film-Noir": B[650]}
ax.scatter(gp.votes, gp.rating, s=gp.films / 9, color=[hl.get(g, B[300]) for g in gp._id], alpha=.85, edgecolor=IVORY, lw=2)
for _, g in gp.iterrows(): ax.annotate(g._id, (g.votes, g.rating), xytext=(0, 9 + np.sqrt(g.films / 9) / 2), textcoords="offset points", ha="center", fontsize=9,
                                       color=INK if g._id in hl else INK2, fontweight="bold" if g._id in hl else None)
ax.set_xscale("log"); ax.set_xlabel("average IMDb votes per film (log)"); ax.set_ylabel("average IMDb rating")
ax.axhline(gp.rating.mean(), color=GRID, lw=1.2)
ax.text(gp.votes.min(), 7.2, "niche, loved", color=MUTED, fontsize=9); ax.text(gp.votes.max(), 5.78, "mass, middling", color=MUTED, fontsize=9, ha="right")
head(ax, "Genres: quality vs reach", "bubble size = number of films; niche genres are rated high by a self-selected audience")
plt.show()

In [ ]:
# critic score (0-10) minus audience score rescaled from 0-5 to 0-10
gap = agg("movies", [{"$match": {"tomatoes.critic.rating": {"$type": "number"}, "tomatoes.viewer.rating": {"$type": "number"}}},
                     {"$unwind": "$genres"},
                     {"$group": {"_id": "$genres", "films": {"$sum": 1}, "critic": {"$avg": "$tomatoes.critic.rating"},
                                 "audience": {"$avg": {"$multiply": ["$tomatoes.viewer.rating", 2]}}}},
                     {"$match": {"films": {"$gte": 100}}}, {"$addFields": {"gap": {"$subtract": ["$critic", "$audience"]}}}, {"$sort": {"gap": 1}}])
# dumbbell: critic dot and audience dot per genre, the line is the disagreement
gs_ = gap.sort_values("gap", ascending=False)  # first row plots at the bottom, so the widest gap ends on top
fig, ax = plt.subplots(figsize=(11, 7))
ax.hlines(gs_._id, gs_.critic, gs_.audience, color=B[150], lw=5, zorder=1)
ax.scatter(gs_.critic, gs_._id, s=80, color=RED, zorder=3, label="critics (RT, 0-10)", edgecolor=IVORY, lw=1.5)
ax.scatter(gs_.audience, gs_._id, s=80, color=B[550], zorder=3, label="audience (RT, rescaled 0-10)", edgecolor=IVORY, lw=1.5)
for _, g in gs_.iterrows(): ax.text(g.audience + .06, g._id, f"{g.gap:+.2f}", va="center", fontsize=8.5, color=MUTED)
ax.grid(axis="y", visible=False); ax.legend(loc="lower left", bbox_to_anchor=(0, -.12), ncol=2, fontsize=9)
head(ax, "Critics vs audiences (dumbbell chart)", "every genre sits right of its critics; the widest gaps are the crowd-pleasers at the top")
plt.show()
gap.round(2)

**Quality vs reach.** The scatter separates genres into clear groups:
- **Niche and high-rated:** Documentary, Short, History, War and Film-Noir have the highest averages but the fewest votes. Their audiences choose them on purpose, and a self-selected audience rates generously.
- **Mass and middling:** Sci-Fi, Adventure, Action and Fantasy pull the most votes per film but sit below average on rating. These are broad-appeal blockbusters seen by many casual viewers.
- **Horror** is the outlier at the bottom, with high reach but the lowest rating of any genre.

**Who disagrees most?** In the dumbbell chart every genre's critic dot sits to the left of its audience dot, so critics score lower than audiences in every genre, but the gap is uneven. It's widest for **Action, Fantasy, Family and Comedy** (about 0.8 to 0.9 points on a 10-point scale), the crowd-pleasing genres critics tend to dismiss. It's narrowest for **Western, War, History and Documentary**, where critics and audiences mostly agree. For a streaming service, this means critic scores are a weak guide to what users will enjoy in exactly the genres people watch most.

### 4. Text analysis: phrases that define each genre

Plots and titles are real text, so the text analysis uses them. Comments are left out because their text is lorem ipsum (confirmed below). For each genre I compare how often a phrase appears in that genre's plots with how often it appears in all other plots (log-odds ratio). This brings up phrases that are *characteristic* of the genre rather than just frequent, so generic words like "life" or "family" drop out.

In [ ]:
# pull plots and titles once, then do the NLP in Python
mv = pd.DataFrame(db.movies.find({"fullplot": {"$type": "string"}, "genres.0": {"$exists": True}},
                                 {"title": 1, "fullplot": 1, "genres": 1, "year": 1, "imdb.rating": 1}))
mv = mv.drop_duplicates(subset=["title", "fullplot"])  # drop the duplicate documents found in part (a)
mv["year"] = pd.to_numeric(mv.year.astype(str).str[:4], errors="coerce")
mv["rating"] = pd.to_numeric(mv.imdb.str["rating"], errors="coerce")
G = ["Drama", "Comedy", "Romance", "Crime", "Thriller", "Action", "Horror", "Documentary", "Sci-Fi", "War", "Western", "Family"]

# log-odds of a phrase appearing in genre g vs every other genre
def distinctive(texts, ngram, min_df, k=10):
    cv = CountVectorizer(ngram_range=ngram, stop_words="english", min_df=min_df)
    X = cv.fit_transform(texts); vocab = np.array(cv.get_feature_names_out()); out = {}
    for g in G:
        m = mv.genres.apply(lambda x: g in x).values
        a, b = np.asarray(X[m].sum(0)).ravel() + 1, np.asarray(X[~m].sum(0)).ravel() + 1
        lo = np.log(a / a.sum()) - np.log(b / b.sum()); lo[a < 6] = -np.inf
        top = np.argsort(-lo)[:k]
        out[g] = ", ".join(vocab[top[np.isfinite(lo[top])]])
    return pd.Series(out)

print(f"movies with a full plot and genre: {len(mv)}")
phr = distinctive(mv.fullplot, (2, 3), 15)
pd.DataFrame({"plot phrases (2-3 grams)": phr}).style.set_properties(**{"text-align": "left"})

In [ ]:
# small multiples: the six most distinctive phrases for six genres, length = log-odds vs all other genres
cv = CountVectorizer(ngram_range=(2, 3), stop_words="english", min_df=15)
X = cv.fit_transform(mv.fullplot); vocab = np.array(cv.get_feature_names_out())
fig, axs = plt.subplots(2, 3, figsize=(14, 7.5))
for ax, g in zip(axs.flat, ["Crime", "Thriller", "Horror", "Romance", "Sci-Fi", "Documentary"]):
    msk = mv.genres.apply(lambda x: g in x).values
    a, b = np.asarray(X[msk].sum(0)).ravel() + 1, np.asarray(X[~msk].sum(0)).ravel() + 1
    lo = np.log(a / a.sum()) - np.log(b / b.sum()); lo[a < 6] = -np.inf
    top_ = np.argsort(-lo)[:7][::-1]
    ax.hlines(vocab[top_], 0, lo[top_], color=B[200], lw=2.5); ax.scatter(lo[top_], vocab[top_], s=55, color=B[600], zorder=3)
    ax.grid(axis="y", visible=False); ax.set_xlim(0, lo[top_].max() * 1.15); ax.tick_params(axis="y", labelsize=9.5, colors=INK2)
    ax.set_title(g, loc="left", fontsize=11.5, fontweight="bold", color=INK)
fig.suptitle("What each genre sounds like", x=.01, ha="left", fontsize=14, fontweight="bold", y=1.02)
fig.text(.01, .975, "phrase log-odds in the genre's plots vs every other genre (longer = more characteristic)", fontsize=9.5, color=MUTED)
plt.tight_layout(); plt.show()

In [ ]:
# same method on titles, single words, to see how films brand themselves
pd.DataFrame({"title words": distinctive(mv.title, (1, 1), 5, 8)}).style.set_properties(**{"text-align": "left"})

In [ ]:
# phrases also drift over time: top plot bigrams that rose or fell between pre-1970 and post-2000 films
cv = CountVectorizer(ngram_range=(2, 2), stop_words="english", min_df=20)
X = cv.fit_transform(mv.fullplot); v = np.array(cv.get_feature_names_out())
old, new = (mv.year < 1970).values, (mv.year >= 2000).values
a, b = np.asarray(X[old].sum(0)).ravel() + 1, np.asarray(X[new].sum(0)).ravel() + 1
lo = np.log(b / b.sum()) - np.log(a / a.sum())
up, dn = np.argsort(-lo)[:12], np.argsort(lo)[:12]
dd = pd.DataFrame({"p": np.r_[v[dn], v[up]], "lo": np.r_[lo[dn], lo[up]]})
fig, ax = plt.subplots(figsize=(11, 8))
ax.hlines(dd.p, 0, dd.lo, color=[RED if x < 0 else B[550] for x in dd.lo], lw=2.5, alpha=.45)
ax.scatter(dd.lo, dd.p, s=60, color=[RED if x < 0 else B[550] for x in dd.lo], zorder=3)
ax.axvline(0, color=MUTED, lw=1); ax.grid(axis="y", visible=False)
ax.text(dd.lo.max(), 23.6, "more common after 2000", ha="right", color=B[550], fontsize=9.5, fontweight="bold")
ax.text(dd.lo.min(), -.6, "more common before 1970", ha="left", color=RED, fontsize=9.5, fontweight="bold")
head(ax, "How plot language drifted", "bigram log-odds, post-2000 plots vs pre-1970 plots")
plt.show()

In [ ]:
# comments as text: vocabulary check against plain English
cv = CountVectorizer(stop_words="english", max_features=15)
cv.fit([d["text"] for d in db.comments.find({}, {"text": 1}).limit(5000)])
print("most frequent comment words:", ", ".join(cv.get_feature_names_out()))

**What the phrases reveal.** The lollipop panels and the table show that each genre has a recognisable vocabulary, and it reads almost like a pitch formula:
- **Crime** is built around institutions: *homicide detective, police department, organized crime, mob boss, bank robber*.
- **Thriller** is built around escalation and time pressure: *goes horribly wrong, cat and mouse game, time running out, serial killer*.
- **Horror** follows a set script: *college students*, a *haunted house*, *strange things* happen, someone *decides to investigate* and ends up *trapped inside* with a *dark secret*.
- **Romance** follows the arc of a relationship: *blind date, mutual attraction, unrequited love, getting married*.
- **Sci-Fi** places itself in time: *near future, distant future, post apocalyptic, time machine*.
- **Documentary** describes how it was made: *unprecedented access, archival footage, intimate portrait*.
- **Action** and **Thriller** share *james bond* and *cia operative*, which shows how much the spy film sits between the two.

Titles tell a different story. Action, Sci-Fi, Horror and Family titles are dominated by **franchise names** (*Batman, Transformers, Terminator, Hellraiser, Elm Street, Barbie, Pooh, Harry Potter*), while Drama titles use evocative, abstract words (*passion, autumn, youth, childhood*). Genre films sell a known brand, while dramas sell a mood.

Over time, plot language shifts from formal, period settings (*turn of the century, high society, upper class, wants to marry*) to everyday modern life (*single mother, road trip, hip hop, group of friends, rock band*). Stories moved from class and marriage toward personal and urban themes.

Finally, the most frequent comment words are *dolor, eius, eos, esse...*, which confirms that the comment text is filler and has no meaning to analyze.

## Part (c): Topic modeling on plot descriptions

Genres are labels assigned by people. Topic modeling asks what themes the plot text itself contains. I use **NMF on TF-IDF features** (unigrams and bigrams) of the ~21k full plot descriptions. NMF tends to give sharper, easier-to-read topics than LDA on short texts like these, and I fit LDA afterwards as a check. I added a domain stopword list because words like *young, man, life, story, finds* appear in nearly every plot and would otherwise dominate every topic. The reconstruction error drops smoothly without a sharp elbow, so I chose **k = 14**: it's the smallest k that separates a sci-fi/adventure theme from the domestic ones while keeping each topic readable.

In [ ]:
# domain stopwords: words every plot uses, which would otherwise dominate every topic
extra = ("young man woman life story film films movie old year years new time day finds find takes take make makes gets get "
         "help way set begins tries discovers named called people lives world end comes come goes go just like turns soon decides "
         "ii 10 11 12 13 14 15 16 17 18 19 20 30").split()
tfidf = TfidfVectorizer(stop_words=list(ENGLISH_STOP_WORDS | set(extra)), max_df=0.3, min_df=10, ngram_range=(1, 2), max_features=30000)
T = tfidf.fit_transform(mv.fullplot); words = tfidf.get_feature_names_out()

# pick k by reconstruction error elbow, then fit the final model
errs = {k: NMF(k, init="nndsvda", random_state=42, max_iter=300).fit(T).reconstruction_err_ for k in [6, 8, 10, 12, 14, 16]}
fig, ax = plt.subplots(figsize=(8, 3.6))
ax.plot(list(errs), list(errs.values()), color=B[500], lw=2, marker="o", ms=8, mec=IVORY, mew=2)
ax.axvline(14, color=RED, lw=1); ax.text(14.2, max(errs.values()), "chosen k = 14", color=RED, fontsize=9, va="top")
ax.set_xlabel("number of topics k"); ax.set_ylabel("reconstruction error")
head(ax, "Choosing k", "error falls smoothly with no sharp elbow, so k is chosen for readability")
plt.show()

In [ ]:
# final NMF with 14 topics, top terms per topic
K = 14
nmf = NMF(K, init="nndsvda", random_state=42, max_iter=500)
W = nmf.fit_transform(T)
terms = pd.DataFrame({f"T{i}": words[c.argsort()[-10:][::-1]] for i, c in enumerate(nmf.components_)}).T
terms.columns = [f"w{j+1}" for j in range(10)]
terms

The 14 topics are easy to read, so I gave each one a name (next cell). They fall into three groups: **plot engines** (police and murder, war, missions to save the world), **relationships** (falling in love, marriage and affairs, mothers, fathers and sons, family and home, friends) and **settings** (high school, small town, big city). The music and documentary topic is about the film's form rather than its story.

In [ ]:
# my labels for the 14 topics after reading the top terms
labels = {0: "Friends, nightlife, money", 1: "Music and documentary", 2: "High school", 3: "Falling in love",
          4: "War and soldiers", 5: "Small-town life", 6: "Family and home", 7: "Fathers and sons",
          8: "Police and murder", 9: "Marriage and affairs", 10: "Missions to save the world",
          11: "Mothers and single parents", 12: "Big-city life", 13: "Childhood and coming of age"}
mv["topic"] = pd.Series(W.argmax(1)).map(labels).values
mv["strength"] = W.max(1)
GROUP = {"Police and murder": 0, "War and soldiers": 0, "Missions to save the world": 0, "Falling in love": 1, "Marriage and affairs": 1,
         "Mothers and single parents": 1, "Fathers and sons": 1, "Family and home": 1, "Friends, nightlife, money": 1,
         "Childhood and coming of age": 1, "High school": 2, "Small-town life": 2, "Big-city life": 2, "Music and documentary": 2}
GNAME = ["plot engines", "relationships", "settings and form"]
# topic cards: top terms per topic, colored by theme family
fig, axs = plt.subplots(2, 7, figsize=(17, 6.4))
for ax, (i, lab) in zip(axs.flat, labels.items()):
    c = nmf.components_[i]; idx = c.argsort()[-6:]
    ax.barh(words[idx], c[idx], height=.5, color=CAT[GROUP[lab]]); ax.set_xticks([]); ax.grid(False)
    ax.tick_params(axis="y", labelsize=8.5, colors=INK2, length=0); ax.set_title(lab, loc="left", fontsize=9.5, fontweight="bold", color=INK, wrap=True)
fig.suptitle("14 themes found in 21k plot descriptions", x=.01, ha="left", fontsize=14, fontweight="bold", y=1.03)
fig.text(.01, .985, "blue = plot engines, orange = relationships, aqua = settings and form; bars = term weight in the NMF topic", fontsize=9.5, color=MUTED)
plt.tight_layout(); plt.show()
summary = mv.groupby("topic").agg(films=("title", "size"), median_year=("year", "median"), avg_rating=("rating", "mean")).round(2)
summary.sort_values("films", ascending=False)

In [ ]:
# Sankey: how each genre's films distribute over the 14 themes (each film counted once per genre)
ex = mv.explode("genres").reset_index(drop=True)
fl = ex[ex.genres.isin(G)].groupby(["genres", "topic"]).size().reset_index(name="n")
fl = fl[fl.n >= 60]
gl, tl = G, sorted(labels.values(), key=lambda t: (GROUP[t], t))
nodes = gl + tl; idx = {n: i for i, n in enumerate(nodes)}
rgba = lambda h, a: f"rgba({int(h[1:3], 16)},{int(h[3:5], 16)},{int(h[5:7], 16)},{a})"
fig = go.Figure(go.Sankey(arrangement="snap",
    node=dict(label=nodes, pad=9, thickness=12, line=dict(width=0), color=[B[650]] * len(gl) + [CAT[GROUP[t]] for t in tl]),
    link=dict(source=[idx[a] for a in fl.genres], target=[idx[b] for b in fl.topic], value=fl.n, color=[rgba(CAT[GROUP[t]], .28) for t in fl.topic])))
fig.update_layout(height=720, paper_bgcolor=IVORY, font=dict(size=11, color=INK, family="DejaVu Sans"), margin=dict(l=10, r=10, t=70, b=10),
                  title=dict(text="<b>From genre to theme</b><br><span style='font-size:12px;color:#898781'>flows of 60+ films; flow color = theme family (blue plot engines, orange relationships, aqua settings)</span>", x=.01))
fig.show()

In [ ]:
# lift = share of a genre's films in a topic divided by the topic's overall share, on a log2 scale and clustered
base = mv.topic.value_counts(normalize=True)
lift = pd.crosstab(ex.topic, ex.genres, normalize="columns")[G].div(base, axis=0)
cg = sns.clustermap(np.log2(lift.clip(lower=.1)), cmap=DIV, center=0, vmin=-3, vmax=3, annot=lift.round(1), fmt="", annot_kws={"fontsize": 8.5},
                    linewidths=2, linecolor=IVORY, figsize=(12, 9), dendrogram_ratio=(.1, .12), cbar_pos=None, tree_kws={"colors": MUTED, "linewidths": .8})
cg.ax_heatmap.set_xlabel(""); cg.ax_heatmap.set_ylabel(""); cg.ax_heatmap.tick_params(length=0)
cg.fig.suptitle("Topic to genre lift, clustered", x=.02, ha="left", y=1.04, fontsize=14, fontweight="bold")
cg.fig.text(.02, 1.005, "numbers = lift (1.0 = no association), blue = genre over-uses the theme, red = avoids it; trees group similar rows and columns", fontsize=9.5, color=MUTED)
plt.show()

In [ ]:
# topic prevalence by era
mv["era"] = pd.cut(mv.year, [1900, 1960, 1980, 2000, 2020], labels=["pre-1960", "1960-79", "1980-99", "2000+"])
era = pd.crosstab(mv.topic, mv.era, normalize="columns") * 100
# slope chart: theme share before 1960 vs after 2000, label positions nudged apart so they never collide
def spread(vals, gap):
    order = np.argsort(vals); out = np.array(vals, float)
    for k_ in range(1, len(order)):
        if out[order[k_]] - out[order[k_ - 1]] < gap: out[order[k_]] = out[order[k_ - 1]] + gap
    return out
sl = era.drop(index="Missions to save the world")  # the catch-all topic would squash the scale
a_, b_ = sl["pre-1960"], sl["2000+"]
fig, ax = plt.subplots(figsize=(10, 10))
la, lb = spread(a_.values, .5), spread(b_.values, .5)
for i, t in enumerate(sl.index):
    c_ = B[550] if b_[t] - a_[t] > .5 else RED if b_[t] - a_[t] < -.5 else "#c9c7bf"
    ax.plot([0, 1], [a_[t], b_[t]], color=c_, lw=2.4 if c_ != "#c9c7bf" else 1.4, marker="o", ms=7, mec=IVORY, mew=1.5, zorder=3)
    ax.plot([-.02, -.1], [a_[t], la[i]], color=GRID, lw=1); ax.plot([1.02, 1.1], [b_[t], lb[i]], color=GRID, lw=1)  # leader lines
    ax.text(-.12, la[i], f"{t}  {a_[t]:.1f}%", ha="right", va="center", fontsize=9, color=INK2)
    ax.text(1.12, lb[i], f"{b_[t]:.1f}%  {t}", ha="left", va="center", fontsize=9, color=c_ if c_ != "#c9c7bf" else INK2, fontweight="bold" if c_ != "#c9c7bf" else None)
ax.set_xlim(-1.0, 2.0); ax.set_xticks([0, 1], ["before 1960", "2000 and later"]); ax.set_yticks([]); ax.grid(False)
for s_ in ax.spines.values(): s_.set_visible(False)
head(ax, "Which themes rose and fell", "share of plots per theme; blue = grew, red = shrank (more than half a point); the catch-all missions topic (18% to 22%) is left out")
plt.show()
era.round(1).sort_values("2000+", ascending=False)

In [ ]:
# plot galaxy: t-SNE of each film's 14 topic weights; nearby films tell similar stories
rng = np.random.default_rng(42); pick = rng.choice(len(mv), size=min(7000, len(mv)), replace=False)
Wn = W[pick] / (W[pick].sum(1, keepdims=True) + 1e-9)
emb = TSNE(2, perplexity=45, init="pca", random_state=42, learning_rate="auto").fit_transform(Wn)
tp = mv.topic.values[pick]
fig, ax = plt.subplots(figsize=(13, 11))
for gi in range(3):
    m_ = np.array([GROUP[t] == gi for t in tp])
    ax.scatter(emb[m_, 0], emb[m_, 1], s=5, color=CAT[gi], alpha=.55, lw=0, label=GNAME[gi])
for t in labels.values():
    m_ = tp == t
    if m_.sum() > 30:
        cx, cy = np.median(emb[m_], axis=0)
        ax.text(cx, cy, t, ha="center", va="center", fontsize=9.5, fontweight="bold", color=INK,
                bbox=dict(boxstyle="round,pad=.3", fc=IVORY, ec=CAT[GROUP[t]], lw=1.2, alpha=.92))
ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
for s_ in ax.spines.values(): s_.set_visible(False)
ax.legend(loc="lower left", markerscale=4, fontsize=9.5)
head(ax, "The plot galaxy", "7,000 films placed by t-SNE on their topic mix; each cloud is a theme, bridges are films that blend two themes")
plt.show()

In [ ]:
# the most representative film per topic, which helps sanity-check the labels
mv.sort_values("strength", ascending=False).groupby("topic").head(2)[["topic", "title", "year", "genres"]].sort_values("topic")

In [ ]:
# LDA on raw counts as a robustness check against NMF
cvl = CountVectorizer(stop_words=list(ENGLISH_STOP_WORDS | set(extra)), max_df=0.3, min_df=10, max_features=15000)
C = cvl.fit_transform(mv.fullplot); lw = cvl.get_feature_names_out()
lda = LatentDirichletAllocation(K, random_state=42, learning_method="online", batch_size=1024, max_iter=10).fit(C)
for i, c in enumerate(lda.components_): print(f"LDA {i:2d}:", ", ".join(lw[c.argsort()[-8:][::-1]]))

**Interpretation.**

*Topics line up with genres, but they don't simply duplicate them.* The Sankey diagram shows where each genre's films go: Crime pours mostly into police and murder, War into war and soldiers, and Drama spreads across almost every theme. The clustered lift heatmap makes this exact. It shows strong matches where you'd expect them: War and soldiers is 10x over-represented in War films, Music and documentary 6.4x in Documentaries, Small-town life 5x in Westerns, and Police and murder 4.3x in Crime. That the model recovers genre without ever seeing the labels suggests the genre tags really do reflect what the stories are about.

The more interesting results are the ones that cut across genres:
- **Police and murder** is also strongly over-represented in Thriller (2.7x) and Action (2.1x), so the investigation plot drives three genres rather than one.
- **Missions to save the world** links Sci-Fi (3.0x), Action (2.1x) and Horror (1.6x). It's also the largest topic, because NMF assigns plots with no strong theme to it, so its size overstates how common the theme is.
- **Small-town life** shows up in Horror (1.6x) as well as Westerns. The isolated community is a shared setting for both genres.
- **War and soldiers** also appears in Westerns (1.9x), mostly Civil War stories.
- **Family** films don't have a topic of their own. They spread across childhood, high school, fathers and sons, and family and home, so "Family" describes the intended audience rather than a kind of story.
- **Drama** is close to 1.0 everywhere. It's the default label rather than a theme.

*The plot galaxy.* Placing each film by its topic mix with t-SNE turns the themes into a map. The relationship themes (orange) sit next to each other in one large region, because family, marriage and love stories blend into each other. Police and murder, war, and missions to save the world (blue) form separate islands, because those plots are built around one dominant engine. The settings (aqua) sit between them, since a high school or a small town can host any kind of story.

*Themes over time.* The slope chart compares the share of each theme before 1960 and after 2000. **Marriage and affairs** (12% to 7%), **falling in love** (11% to 7%) and **war** (10% to 4%) have all declined since before 1960, while **music and documentary** (4% to 9%), **high school** (2% to 5%) and **mothers and single parents** (3.5% to 6%) have grown. Classic cinema was built around courtship, marriage and war. Modern cinema spends more time on adolescence, single-parent families and real-world subjects, which matches the shift in plot phrases in part (b).

*Ratings by topic.* War and soldiers and Music and documentary have the highest average ratings (about 7.0), and High school, Friends and nightlife, and Missions to save the world have the lowest (about 6.55). This is the same pattern as the genre analysis: serious, niche themes are rated higher by their chosen audiences, and mass-market youth and spectacle themes are rated lower.

*LDA check.* LDA finds the same main themes (war, police and murder, documentary and music, space and planets, school, family), which suggests the structure is real and not a quirk of NMF. Its topics are noisier, though, and full of character names (*jack, alex, max, johnny*), because LDA works on raw counts and frequent first names get through. That's why I report NMF as the main model.

## Summary
1. **The catalogue is real; the engagement layer is not.** Movie metadata from IMDb and Rotten Tomatoes is rich and mostly reliable. The users and comments are synthetic: comments are lorem ipsum, 63% predate the films they review, and their volume ignores popularity. Recognising this kept me from building conclusions on fake data.
2. **A document store needs analyst-side validation.** Mixed types, empty strings and duplicate documents all exist because nothing enforces a schema, so filtering by `$type` was necessary in every pipeline.
3. **Older films rate higher mostly because of survivorship,** not because cinema has declined. Horror fell the furthest, while Documentary rose both in share and in rating.
4. **Critics and audiences are two different systems** (r = 0.65). Their disagreement is largest in the crowd-pleasing genres (Action, Fantasy, Comedy, Family), which is where audience-based recommendations matter most.
5. **Genres have recognisable phrase patterns**, and modeled topics recover genres without labels. Topics also show that story types cross genre lines (investigation drives Crime, Thriller and Action; isolated towns serve both Westerns and Horror) and that cinema has moved from courtship, marriage and war toward adolescence, family and documentary.